<a href="https://colab.research.google.com/github/adithi2307/IN126055302_GEN_AI/blob/main/TASK_2_SENTIMENT_ANALYSIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ==============================
# TASK 2: SENTIMENT ANALYSIS NLP PIPELINE
# ==============================

import re
import pandas as pd
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ==============================
# STEP 1: PREPROCESS FUNCTION
# ==============================

def preprocess_text(text):
    if not text or not isinstance(text, str):
        return ""

    # Remove URLs & emails
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Handle repeated characters
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(r"[^\w\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    # Remove short tokens ≤2 except 'no', 'not'
    tokens = [w for w in tokens if len(w) > 2 or w in ['no', 'not']]

    return " ".join(tokens)


# ==============================
# STEP 2: DATASET
# ==============================

data = {
    "text": [
        "I love this product",
        "This is the worst thing ever",
        "Amazing experience",
        "Not good at all",
        "Very happy with the service",
        "I hate this",
        "Absolutely fantastic!",
        "Terrible experience",
        "I am not satisfied",
        "Best purchase ever",
        "Worst quality product",
        "I really enjoyed this",
        "This is awful",
        "Highly recommend this",
        "Not worth the money"
    ],
    "label": [
        1, 0, 1, 0, 1, 0,
        1, 0, 0, 1, 0, 1,
        0, 1, 0
    ]
}

df = pd.DataFrame(data)

print("\n===== ORIGINAL DATA =====\n")
print(df)


# ==============================
# STEP 3: APPLY PREPROCESSING
# ==============================

df["clean_text"] = df["text"].apply(preprocess_text)

print("\n===== CLEANED DATA =====\n")
print(df[["text", "clean_text"]])


# ==============================
# STEP 4: TEXT → NUMBERS (TF-IDF)
# ==============================

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["clean_text"])
y = df["label"]

print("\nTF-IDF Shape:", X.shape)


# ==============================
# STEP 5: TRAIN TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTraining size:", X_train.shape)
print("Testing size:", X_test.shape)


# ==============================
# STEP 6: TRAIN MODEL
# ==============================

model = LogisticRegression()
model.fit(X_train, y_train)

print("\nModel training completed")


# ==============================
# STEP 7: EVALUATION
# ==============================

y_pred = model.predict(X_test)

print("\n===== MODEL EVALUATION =====\n")

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


# ==============================
# STEP 8: TEST CUSTOM INPUTS
# ==============================

print("\n===== CUSTOM TEST CASES =====\n")

test_sentences = [
    "I really love this!",
    "This is absolutely terrible",
    "Not bad at all",
    "Worst experience ever!!!",
    "I am very happy",
    "I do not like this product"
]

# preprocess
clean_test = [preprocess_text(t) for t in test_sentences]

# transform
test_vec = vectorizer.transform(clean_test)

# predict
predictions = model.predict(test_vec)

for text, pred in zip(test_sentences, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"{text} → {sentiment}")


# ==============================
# STEP 9: FREQUENCY ANALYSIS (BONUS)
# ==============================

print("\n===== WORD FREQUENCY =====\n")

all_words = " ".join(df["clean_text"]).split()
word_counts = Counter(all_words)

print("Top 10 Words:", word_counts.most_common(10))


print("TASK COMPLETED SUCCESSFULLY")


===== ORIGINAL DATA =====

                            text  label
0            I love this product      1
1   This is the worst thing ever      0
2             Amazing experience      1
3                Not good at all      0
4    Very happy with the service      1
5                    I hate this      0
6          Absolutely fantastic!      1
7            Terrible experience      0
8             I am not satisfied      0
9             Best purchase ever      1
10         Worst quality product      0
11         I really enjoyed this      1
12                 This is awful      0
13         Highly recommend this      1
14           Not worth the money      0

===== CLEANED DATA =====

                            text                   clean_text
0            I love this product            love this product
1   This is the worst thing ever    this the worst thing ever
2             Amazing experience           amazing experience
3                Not good at all                 not good

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_